NOTEBOOK 4: LoRA — Llama-3.2-3B  +  QLoRA — Phi-3.5-mini
Account : 4  |  GPU: T4 x1  |  Expected: ~3.5 hrs
BEFORE RUNNING:
 1. Kaggle Secrets → HF_TOKEN (for Llama — accept license at hf.co)
 2. GPU: T4 x1  |  Internet: ON

In [ ]:
!pip install -q -U transformers datasets accelerate peft trl bitsandbytes scikit-learn

In [ ]:
import pandas as pd
import numpy as np
import json, os, time, gc, warnings
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
    TrainingArguments, set_seed
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
from trl import SFTTrainer

warnings.filterwarnings('ignore')
set_seed(42)

try:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login
    login(token=UserSecretsClient().get_secret("HF_TOKEN"))
    print("✅ HF login OK")
except Exception as e:
    print(f"⚠️  HF login: {e}")

LABEL2ID = {'positive': 0, 'negative': 1, 'neutral': 2}
CSV_PATH = '/kaggle/input/sentfin/SEntFiN-v1_1.csv'
SAVE_DIR = '/kaggle/working'
RESULTS  = []

print(f"GPU : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB" if torch.cuda.is_available() else "")

In [ ]:
def preprocess_data(csv_path, seed=42):
    df = pd.read_csv(csv_path)
    rows = []
    for _, row in df.iterrows():
        try:
            for entity, label in json.loads(row['Decisions']).items():
                if label in LABEL2ID:
                    rows.append({'headline': str(row['Title']).strip(),
                                 'entity':   str(entity).strip(),
                                 'label':    label})
        except Exception:
            continue
    data = pd.DataFrame(rows)
    print(f"Exploded: {len(data)} | {data['label'].value_counts().to_dict()}")
    train, temp = train_test_split(data, test_size=0.2, random_state=seed, stratify=data['label'])
    val,  test  = train_test_split(temp, test_size=0.5, random_state=seed, stratify=temp['label'])
    return train.reset_index(drop=True), val.reset_index(drop=True), test.reset_index(drop=True)

train_df, val_df, test_df = preprocess_data(CSV_PATH)

In [ ]:
SYSTEM_PROMPT = (
    "You are a financial sentiment analyst. Given a news headline and a company name, "
    "classify the sentiment towards that company. "
    "Reply with exactly one word: positive, negative, or neutral."
)

def make_training_text(headline, entity, label, tokenizer):
    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": f"Headline: {headline}\nCompany: {entity}\nSentiment:"},
        {"role": "assistant", "content": label}
    ]
    try:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    except Exception:
        msgs = [
            {"role": "user",      "content": f"{SYSTEM_PROMPT}\n\nHeadline: {headline}\nCompany: {entity}\nSentiment:"},
            {"role": "assistant", "content": label}
        ]
        return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)

def make_inference_prompt(headline, entity, tokenizer):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": f"Headline: {headline}\nCompany: {entity}\nSentiment:"}
    ]
    try:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    except Exception:
        msgs = [{"role": "user",
                 "content": f"{SYSTEM_PROMPT}\n\nHeadline: {headline}\nCompany: {entity}\nSentiment:"}]
        return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def extract_label(text):
    t = text.lower().strip()
    for lbl in ['positive', 'negative', 'neutral']:
        if t.startswith(lbl) or lbl in t:
            return lbl
    return 'neutral'

def evaluate_generative(model, tokenizer, df, split_label, batch_size=8):
    model.eval()
    tokenizer.padding_side = 'left'
    all_preds, all_labels = [], df['label'].tolist()
    t0 = time.time()

    for i in range(0, len(df), batch_size):
        b = df.iloc[i:i+batch_size]
        prompts = [make_inference_prompt(r['headline'], r['entity'], tokenizer) for _, r in b.iterrows()]
        inputs  = tokenizer(prompts, return_tensors='pt', padding=True,
                             truncation=True, max_length=120).to('cuda')
        with torch.no_grad():
            out = model.generate(
                **inputs, max_new_tokens=5, do_sample=False,
                pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id
            )
        il = inputs['input_ids'].shape[1]
        for seq in out:
            all_preds.append(extract_label(tokenizer.decode(seq[il:], skip_special_tokens=True)))

        if i % (batch_size * 20) == 0:
            print(f"  Eval: {min(i+batch_size, len(df))}/{len(df)}")

    ms     = (time.time() - t0) * 1000 / len(df)
    pred_i = [LABEL2ID.get(p, 2) for p in all_preds]
    lab_i  = [LABEL2ID.get(l, 2) for l in all_labels]
    acc    = accuracy_score(lab_i, pred_i)
    mf1    = f1_score(lab_i, pred_i, average='macro', zero_division=0)
    per_f1 = f1_score(lab_i, pred_i, average=None, labels=[0,1,2], zero_division=0)

    print(f"\n{'='*50}\n[{split_label}] Acc={acc:.4f} MacroF1={mf1:.4f}")
    print(f"  Pos={per_f1[0]:.4f} | Neg={per_f1[1]:.4f} | Neu={per_f1[2]:.4f} | {ms:.1f}ms/sample")
    print(classification_report(lab_i, pred_i,
          target_names=['positive','negative','neutral'], zero_division=0))
    return {'model': split_label, 'accuracy': round(acc,4), 'macro_f1': round(mf1,4),
            'pos_f1': round(per_f1[0],4), 'neg_f1': round(per_f1[1],4),
            'neu_f1': round(per_f1[2],4), 'inference_ms': round(ms,2)}

In [ ]:
def train_lora(model_path, short_name):
    print(f"\n{'='*60}\nLoRA: {short_name}\n{'='*60}")
    tok = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token; tok.pad_token_id = tok.eos_token_id
    tok.padding_side = 'right'

    model = AutoModelForCausalLM.from_pretrained(
        model_path, torch_dtype=torch.bfloat16, device_map='auto', trust_remote_code=True
    )

    targets = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]
    model = get_peft_model(model,
        LoraConfig(r=16, lora_alpha=32, target_modules=targets,
                   lora_dropout=0.05, bias='none', task_type=TaskType.CAUSAL_LM))
    model.print_trainable_parameters()

    texts = [{'text': make_training_text(r['headline'], r['entity'], r['label'], tok)}
             for _, r in train_df.iterrows()]

    SFTTrainer(
        model=model,
        args=TrainingArguments(
            output_dir=f'{SAVE_DIR}/{short_name}',
            num_train_epochs=3,
            per_device_train_batch_size=4, gradient_accumulation_steps=8,
            learning_rate=2e-4, lr_scheduler_type='cosine', warmup_ratio=0.03,
            fp16=False, bf16=True, logging_steps=25, save_strategy='no',
            optim='adamw_torch', report_to='none',
            gradient_checkpointing=True,
            gradient_checkpointing_kwargs={'use_reentrant': False},
        ),
        train_dataset=Dataset.from_list(texts),
        tokenizer=tok, dataset_text_field='text', max_seq_length=128, packing=False,
    ).train()

    model.save_pretrained(f'{SAVE_DIR}/{short_name}_adapter')
    tok.save_pretrained(f'{SAVE_DIR}/{short_name}_adapter')

    vr = evaluate_generative(model, tok, val_df,  f'{short_name}_val')
    tr = evaluate_generative(model, tok, test_df, f'{short_name}_test')

    del model, tok; gc.collect(); torch.cuda.empty_cache()
    print(f"✅ {short_name} done")
    return vr, tr

In [ ]:
def train_qlora(model_path, short_name):
    print(f"\n{'='*60}\nQLoRA: {short_name}\n{'='*60}")
    tok = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token; tok.pad_token_id = tok.eos_token_id
    tok.padding_side = 'right'

    bnb_cfg = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True
    )
    model = AutoModelForCausalLM.from_pretrained(
        model_path, quantization_config=bnb_cfg, device_map='auto', trust_remote_code=True
    )
    model = prepare_model_for_kbit_training(model)

    # Phi-3.5-mini uses fused QKV + gate_up; other models use split projections
    if 'phi' in model_path.lower():
        targets = ["qkv_proj", "o_proj", "gate_up_proj", "down_proj"]
    else:
        targets = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]

    model = get_peft_model(model,
        LoraConfig(r=16, lora_alpha=32, target_modules=targets,
                   lora_dropout=0.05, bias='none', task_type=TaskType.CAUSAL_LM))
    model.print_trainable_parameters()

    texts = [{'text': make_training_text(r['headline'], r['entity'], r['label'], tok)}
             for _, r in train_df.iterrows()]

    SFTTrainer(
        model=model,
        args=TrainingArguments(
            output_dir=f'{SAVE_DIR}/{short_name}',
            num_train_epochs=3,
            per_device_train_batch_size=4, gradient_accumulation_steps=8,
            learning_rate=2e-4, lr_scheduler_type='cosine', warmup_ratio=0.03,
            fp16=False, bf16=True, logging_steps=25, save_strategy='no',
            optim='paged_adamw_32bit', report_to='none',
            gradient_checkpointing=True,
            gradient_checkpointing_kwargs={'use_reentrant': False},
        ),
        train_dataset=Dataset.from_list(texts),
        tokenizer=tok, dataset_text_field='text', max_seq_length=128, packing=False,
    ).train()

    model.save_pretrained(f'{SAVE_DIR}/{short_name}_adapter')
    tok.save_pretrained(f'{SAVE_DIR}/{short_name}_adapter')

    vr = evaluate_generative(model, tok, val_df,  f'{short_name}_val')
    tr = evaluate_generative(model, tok, test_df, f'{short_name}_test')

    del model, tok; gc.collect(); torch.cuda.empty_cache()
    print(f"✅ {short_name} done")
    return vr, tr

In [ ]:
print("\n" + "="*60)
print("MODEL 1: Llama-3.2-3B-Instruct  [LoRA, ~1.5 hrs]")
print("="*60)
vr, tr = train_lora('meta-llama/Llama-3.2-3B-Instruct', 'Llama-3.2-3B')
RESULTS.extend([vr, tr])

In [ ]:
print("\n" + "="*60)
print("MODEL 2: Phi-3.5-mini-instruct  [QLoRA, ~2 hrs]")
print("="*60)
vr, tr = train_qlora('microsoft/Phi-3.5-mini-instruct', 'Phi-3.5-mini')
RESULTS.extend([vr, tr])

In [ ]:
out = pd.DataFrame(RESULTS)
out.to_csv(f'{SAVE_DIR}/results_lora_small2.csv', index=False)
print("\n✅ Saved → /kaggle/working/results_lora_small2.csv")
print(out.to_string(index=False))